In [2]:
import os
import dotenv
import geopandas as gpd
import pandas as pd
import numpy as np
import logging
from datetime import datetime

gpd.options.io_engine = "pyogrio"
os.environ["PYOGRIO_USE_ARROW"] = "1"

dotenv.load_dotenv()
output_parquets=os.getenv('OUTPUT_PARQUETS')
input_parquet=os.getenv("INPUT_PARQUETS")
ret_class=os.path.join(input_parquet,'rec_class_all.parquet')
wrk_dir=os.getenv("WORKING_DIR")
brwmb_targets_csv=os.getenv("BRWMB_TARGETS")
source_data=os.getenv("source_data")
aflb_parquet=os.getenv("ALFB_PARQUET")
cia_dissolve=os.path.join(input_parquet,"cia_dissolve.parquet")
step2d_base=os.path.join(output_parquets,"step_2d_base_aflb.parquet")
table_outputs=os.getenv('TABLE_OUPUTS')
#set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
info=logging.info
debug=logging.debug
error=logging.error

#variables 
output_dir = os.path.join(wrk_dir, "brwmb_parquets")
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    
input_data_gdb=os.path.join(source_data,'Scen_5_inputs.gdb')


import duckdb
conn = duckdb.connect()
conn.install_extension("httpfs")
conn.install_extension("spatial")
conn.install_extension("arrow")
conn.load_extension("spatial")
conn.load_extension("httpfs")
conn.load_extension("arrow")

In [ ]:
rec=gpd.read_parquet(ret_class)


In [4]:
#comment in if for rep short is not calcualted already 
rep_map = {
    'High Productivity Coniferous Leading': 'HPC',
    'High Productivity Deciduous Leading': 'HPD',
    'High Productivity Mixedwood': 'HPM',
    'Medium Productivity Coniferous Leading': 'MPC',
    'Medium Productivity Deciduous Leading': 'MPD',
    'Medium Productivity Mixedwood': 'MPM',
    'Low Productivity Coniferous Leading': 'LPC',
    'Low Productivity Deciduous Leading': 'LPD',
    'Low Productivity Mixedwood': 'LPM',
}
rec['Rec_Cat_short'] = rec['FOR_REP'].map(rep_map)

In [ ]:

rec['aflb_ha']=rec['aflb_fact_bfrn']*(rec.geometry.area/10000)
rec_cat_for_rep=rec.groupby(['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','Rec_Cat_short']).agg(total_aflb_ha=('aflb_ha', 'sum')).reset_index()

In [ ]:

targets_25 = (
    rec
    .groupby(['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat_short'])
    .agg(total_aflb_ha=('aflb_ha','sum'))
    .reset_index()
)
targets_25['total_aflb_ha'] *= 0.25


In [ ]:
#validation cell to explain total area and target area for blueberry river and rec cat/ for rep categories
# wmb = rec_cat_for_rep[
#         (rec_cat_for_rep['NAME'] == 'Blueberry River') #&
#         #(rec_cat_for_rep['Rec_Cat'].isin([1,2,3,4,5]))
#     ]

# target = targets_25[
#         (targets_25['NAME'] == 'Blueberry River')
# ]

# pivot = (
#     wmb
#     .pivot_table(
#         index='Rec_Cat',
#         columns='Rec_Cat_short',
#         values='total_aflb_ha',
#         aggfunc='sum',
#         fill_value=0
#     )
# )
# TOTAL_LABEL = 'Total AFLB (Ha)'
# TARGET_LABEL = '25% of Total'
# SELECTED_LABEL = 'Selected Area (Rec Cat 1-5)'
# EXCLUDED_LABEL = 'Excluded Area (Rec Cat 0)'

# pivot.loc[TOTAL_LABEL] = pivot.sum(numeric_only=True)
# pivot.loc[TARGET_LABEL] = pivot.loc[TOTAL_LABEL] * 0.25

# data = pivot.drop(index=[TOTAL_LABEL, TARGET_LABEL])
# data_for_csum = data.drop(index=0, errors='ignore')

# target_25 = pivot.loc[TARGET_LABEL]

# csum = data_for_csum.cumsum()
# capped_csum = csum.clip(upper=target_25, axis=1)
# allocated_nonzero = capped_csum.diff().fillna(capped_csum)

# result = allocated_nonzero.reindex(data.index).fillna(0)



# final_table = result.copy()
# final_table.loc[TOTAL_LABEL] = pivot.loc[TOTAL_LABEL]
# final_table.loc[TARGET_LABEL] = pivot.loc[TARGET_LABEL]

# selected_area = (
#     final_table
#     .loc[final_table.index.intersection([1, 2, 3, 4, 5])]
#     .sum()
# )

# final_table.loc[SELECTED_LABEL] = selected_area

# excluded_area = data.loc[0] if 0 in data.index else 0
# final_table.loc[EXCLUDED_LABEL] = excluded_area

# final_table

In [ ]:
def creat_rec_tables(
    wmb_name: str,
    rec_cat_for_rep: pd.DataFrame
) -> pd.DataFrame:

    wmb = rec_cat_for_rep[
        (rec_cat_for_rep['WATER_MANAGEMENT_BASIN_NAME'] == wmb_name) #&
        #(rec_cat_for_rep['Rec_Cat'].isin([1,2,3,4,5]))
    ]

    pivot = (
        wmb
        .pivot_table(
            index='Rec_Cat',
            columns='Rec_Cat_short',
            values='total_aflb_ha',
            aggfunc='sum',
            fill_value=0
        )
    )
    TOTAL_LABEL = 'Total AFLB (Ha)'
    TARGET_LABEL = '25% of Total'
    SELECTED_LABEL = 'Selected Area (Rec Cat 1-5)'
    EXCLUDED_LABEL = 'Excluded Area (Rec Cat 0)'

    pivot.loc[TOTAL_LABEL] = pivot.sum(numeric_only=True)
    pivot.loc[TARGET_LABEL] = pivot.loc[TOTAL_LABEL] * 0.25

    data = pivot.drop(index=[TOTAL_LABEL, TARGET_LABEL])
    data_for_csum = data.drop(index=0, errors='ignore')

    target_25 = pivot.loc[TARGET_LABEL]

    csum = data_for_csum.cumsum()
    capped_csum = csum.clip(upper=target_25, axis=1)
    allocated_nonzero = capped_csum.diff().fillna(capped_csum)

    result = allocated_nonzero.reindex(data.index).fillna(0)



    final_table = result.copy()
    final_table.loc[TOTAL_LABEL] = pivot.loc[TOTAL_LABEL]
    final_table.loc[TARGET_LABEL] = pivot.loc[TARGET_LABEL]

    selected_area = (
        final_table
        .loc[final_table.index.intersection([1, 2, 3, 4, 5])]
        .sum()
    )

    final_table.loc[SELECTED_LABEL] = selected_area

    excluded_area = data.loc[0] if 0 in data.index else 0
    final_table.loc[EXCLUDED_LABEL] = excluded_area

    final_table
    return final_table.reset_index()

In [9]:
bbr=creat_rec_tables('Blueberry River', rec_cat_for_rep)
lsc=creat_rec_tables('Lower Sikanni Chief River', rec_cat_for_rep)
mbr=creat_rec_tables('Middle Beatton River', rec_cat_for_rep)
ubr=creat_rec_tables('Upper Beatton River', rec_cat_for_rep)


In [10]:
rec_output_path=os.path.join(table_outputs, f"recruitment_tables_{datetime.now().strftime('%Y-%m-%d')}.xlsx")
with pd.ExcelWriter(rec_output_path, engine="openpyxl") as writer:
    startrow = 0

    for name, df in [
        ("Blueberry River", bbr),
        ("Lower Sikanni Chief River", lsc),
        ("Middle Beatton River", mbr),
        ("Upper Beatton River", ubr),
    ]:
        # Write a title row
        pd.DataFrame([[name]]).to_excel(
            writer,
            sheet_name="Recruitment Tables",
            startrow=startrow,
            index=False,
            header=False
        )

        # Write the table just below the title
        df.to_excel(
            writer,
            sheet_name="Recruitment Tables",
            startrow=startrow + 1,
            index=True
        )

        # Advance startrow (table height + spacing)
        startrow += len(df) + 4
